# Face pass on a free Colab GPU (studio/colab/face-pass.ipynb)

Makes the rendered instructor look more human. Two experiments on a short clip; compare the outputs, then pick one for the full video.

- **A · LatentSync 1.5** re-draws the mouth from the voice audio (real teeth, lips, jaw). Fits a free T4 (needs ~8 GB VRAM; 1.6 needs 18 GB).
- **B · FaceFusion** puts a real face from a photo onto the rendered face (keeps the 3D body and gestures), plus a face enhancer.

**Inputs** (upload in cell 1): `clip.mp4` (the rendered clip), `voice.wav` (its audio), `face.jpg` (only for B).
**Outputs:** `out_A_latentsync.mp4`, `out_B_facefusion.mp4`: download them to `~/Downloads`; the studio picks them up from there.

Runtime → Change runtime type → **T4 GPU**. Licences: check each model's terms before any commercial use; the owner's avatar builds are personal only.


In [ ]:
# 1 · GPU check + inputs (upload, or put them in /content by any other means)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
os.makedirs('/content/in', exist_ok=True); os.chdir('/content/in')
try:
    from google.colab import files
    up = files.upload()            # choose clip.mp4, voice.wav and (for B) face.jpg
except Exception as e:
    print('Upload widget not available here:', e, '\nCopy the files into /content/in another way (e.g. Drive), then continue.')
print(os.listdir('/content/in'))

## A · LatentSync 1.5 (lip-sync from the audio)

In [ ]:
%cd /content
!git clone -q https://github.com/bytedance/LatentSync.git
%cd /content/LatentSync
!pip -q install -r requirements.txt
!pip -q install -U "huggingface_hub[cli]"
!huggingface-cli download ByteDance/LatentSync-1.5 whisper/tiny.pt --local-dir checkpoints
!huggingface-cli download ByteDance/LatentSync-1.5 latentsync_unet.pt --local-dir checkpoints
!ls -la checkpoints checkpoints/whisper

In [ ]:
# 1.5 uses the 256 px UNet config (1.6 is stage2_512.yaml). If the repo renamed it, list configs/unet and pick the 256 one.
%cd /content/LatentSync
!ls configs/unet
!python -m scripts.inference \
    --unet_config_path configs/unet/stage2.yaml \
    --inference_ckpt_path checkpoints/latentsync_unet.pt \
    --inference_steps 25 --guidance_scale 1.5 \
    --video_path /content/in/clip.mp4 --audio_path /content/in/voice.wav \
    --video_out_path /content/out_A_latentsync.mp4
!ls -la /content/out_A_latentsync.mp4

## B · FaceFusion (real face from a photo + enhancer)

In [ ]:
%cd /content
!git clone -q https://github.com/facefusion/facefusion.git
%cd /content/facefusion
!python install.py --onnxruntime cuda --skip-conda
!python facefusion.py --help | head -40

In [ ]:
%cd /content/facefusion
!python facefusion.py headless-run \
    --processors face_swapper face_enhancer \
    --face-swapper-model inswapper_128_fp16 \
    --execution-providers cuda \
    --source-paths /content/in/face.jpg \
    --target-path /content/in/clip.mp4 \
    --output-path /content/out_B_facefusion.mp4
!ls -la /content/out_B_facefusion.mp4

## Download the results

In [ ]:
try:
    from google.colab import files
    for f in ('/content/out_A_latentsync.mp4', '/content/out_B_facefusion.mp4'):
        if os.path.exists(f): files.download(f)
except Exception as e:
    print('Download widget not available here:', e, '\nDownload /content/out_*.mp4 from the file browser.')